In [ ]:
from playwright.async_api import async_playwright

p = await async_playwright().start()
browser = await p.chromium.connect_over_cdp("http://127.0.0.1:9111")
page = await browser.new_page()

In [ ]:
await page.goto("https://snoonu.com");

In [ ]:
# hover the page
await page.hover("body")

In [ ]:
search_results = page.locator('div[class*="SearchResults_group"]')

In [ ]:
merchants = await search_results.locator(">div").all()

In [ ]:
import json
captured_data = {}
page.set_default_navigation_timeout(0)
def handle_route(route):
    print("intercepting")
    response = route.fetch()
    body = response.body()
    data = json.loads(body.decode('utf-8'))
    captured_data['pageProps'] = data.get('pageProps', {})
    route.fulfill(response=response)
await page.route("**/_next/data/**/search.json*", handle_route )


In [57]:
await page.goto("https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017")

<Response url='https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017' request=<Request url='https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017' method='GET'>>

In [58]:
captured_data

{}

In [49]:
from urllib.parse import urlparse
from typing import TypedDict, Optional
from playwright.async_api import Locator
import asyncio
results = []
hostname = urlparse(page.url).hostname
page.set_default_timeout(2000)

class item_result(TypedDict, total=False):
    name: Optional[str]
    image: Optional[str]
    price: Optional[str]
    url: Optional[str]
    
async def get_item_data(item: Locator):
    result: item_result = {}
    try:
        result['image'] = await item.locator("img[class*='CardProductImage']").get_attribute("src")
        print(result['image'])
    except:
        pass
    info = item.locator("div[class*='ProductCartVerticalDescription_info']")
    result['price'] = await info.locator("[class*='price']").text_content()
    result['name'] = await info.locator("[class*='name']").text_content()
    result['url'] = await item.get_attribute("href")
    return result


    
async def get_merchant_data(merchant: Locator):
    
    merchant_info = merchant.locator("div[class*='SearchMerchant_info']")
    merchant_name = await merchant_info.locator("[class*='SearchMerchant_name']").text_content()
    merchant_url = await merchant.locator(">a").get_attribute("href")
    merchant_carousel = merchant.locator("div[class*='SearchMerchant_carousel']")
    items = await merchant_carousel.locator(">div").all()
    
    print(f"Processing {merchant_name}")
    
    return {
        "name": merchant_name,
        "url": merchant_url,
        "items": await asyncio.gather(*
            [get_item_data(item) for item in items]
        )}
    
for m in asyncio.as_completed((
        get_merchant_data(merchant)
        for merchant in await search_results.locator(">div").all()
)):
    await m

Processing Al kheesa Express
Processing Spar Qatar
Processing Dekanet Beirut
Processing Mega Mart
Processing New Family Hypermarket
Processing Snoomart
Processing Carrefour
Processing Al Meera
Processing City Hypermarket
Processing Shaklaan Food Center
Processing Lulu Hypermarket
Processing Arwa Shopping Center
Processing Shopwise Hypermarket
Processing Marks & Spencer
Processing Day And Night
Processing Savemore Market
Processing Duhail Food Center
Processing Torba Store
Processing Monoprix
Processing Natureland


TimeoutError: Locator.text_content: Timeout 2000ms exceeded.
Call log:
  - waiting for locator("div[class*=\"SearchResults_group\"]").locator(">div").nth(15).locator("div[class*='SearchMerchant_carousel']").locator(">div").nth(6).locator("div[class*='ProductCartVerticalDescription_info']").locator("[class*='price']")
